## 2. MateConv的数据收集与数据预处理

### 2.1 MateConv的预训练数据组成

MateConv所使用的语料是由开源数据集和我们自制的数据分析报告/商业分析报告数据集结合、但数据体量要大得多、并且语料也没有那么干净。在构建大型模型的训练预料时，我们有如下三个基本准则——

1. **规模**：构建大型模型的训练语料需要以T级别数据为目标，这意味着数据集的规模必须能够覆盖大模型所需的丰富知识和复杂模式。幸运的是，无论是在中文社区、还是英文社区中，现在都存在许多规模巨大的语料包供我们选择。在训练10B以下模型时，公开语料本身足以覆盖我们的需求。当然，我们要追求的是规模和质量的平衡、单纯增大规模并无意义。

2. **多样性**：不仅包括语料来源的多样性，还应涵盖覆盖领域、语言的多样性以及文本类型的多样性。语料来源应囊括学术文章、新闻报道、社交媒体内容、技术文档等多种形式，以保证模型适应不同的语境。覆盖领域需要广泛，包括科学、技术、艺术、教育、医学和法律等，以确保模型在各个行业都有所表现。同时，语言的多样性要求语料覆盖常见语言和低资源语言，满足多语言模型的训练需求。文本类型的多样性同样重要，应包含叙述性文本、对话文本、代码样本等，从而增强模型在不同应用场景中的表现。

3. **质量**：高质量的数据是训练优秀模型的前提。语料需要经过严格的清洗与筛选，确保其语法正确、内容无噪声且无重复。同时，数据应具备良好的可读性，避免低质量或不完整的文本进入数据集。此外，语料的合法性和道德合规性至关重要，需要确保数据来源符合法律要求，不包含敏感信息或侵犯隐私的内容。高质量的标注和丰富的元信息也能帮助模型更好地理解数据，从而提升训练效果。

在MateConv构建过程中，我们使用了55%中文、37%英文及其他语言、8%代码的方式进行构建，总数据集大小为4.33TB（当然由于每个数据集的存储格式不同、因此数据集所占用的内存大小本身并不完全代表数据量的大小，但依照存储的格式来判断，通常来说数据量是 csv < Parquet < JSON < JSONL）。为了降低训练成本，我们选择了在数据集打包时就已经经过一定清洗的数据、而没有选择完全没清洗过的数据。当然、这些数据还要再经过清洗流程才能够使用、但耗费的时间与人力会远远低于直接在raw data上进行清洗。

| 数据集编号 | 数据集名称                 | 数据属性（中文/英文/代码）                 | 数据量级           | 存储格式            | 是否经过数据清洗 |
|---|----------------------------|-------------------------------------------|--------------------|---------------------|------------------|
| 1| Skywork-SkyPile 150B       | 中文文本                                      | 620GB             | JSONL 短文本        | 是               |
| 2| wanjuan1.0-nlp-CN             | 中文文本                                 | 580GB | JSONL 短文本压缩  | 是               |
| 3| WuDaoCorporaText | 中文文本                                      | 200GB  | JSONL 短文本        | 是               |
| 4| chinese-fineweb-edu-v2 | 75%中文文本，25%英文文本                                      | 670GB  | Parquet        |   是            |
| 5| Wikipedia-CN | 中文文本                                      | 1.1GB  | JSON        |   是            |
| 6| BaiduBaike-5.63M | 中文文本                                      | 17GB  | JSON        |   是            |
| 7| wangrui6/Zhihu-KOL | 中文问答对                                      | 1.5 GB  | parquet        |   是            |
| 8| wanjuan1.0-nlp-EN             | 英文文本                                 | 440GB | JSONL 短文本压缩  | 是               |
| 9| SlimPajama-627B            | 英文为主的混合语言文本                                      | 1TB               | JSONL 短文本压缩    | 是               |
| 10| Starcoder                  | 代码             | 100+GB/768G             | Parquet            | 是               |
| 11| TheStackDedup                 | 代码             | 700+GB/3TB             | Parquet            | 是               |

在这些数据中、大部分为纯粹的文本数据、有很少的一部分为问答对数据。通常来说，我们不会在预训练阶段加入问答对，但业内有研究声称在预训练阶段加入问答对可以提升模型表现。出于教学目的（教导大家如何处理问答对）以及验证目的（观察是否在预训练阶段加入问答对可以提升模型表现），我们保留了少许问答对数据。同时，除了大型的数据之外，我们还使用了10G上下、1G上下的不同大小的文字数据，如果你没有足够的硬件设备可以拉取这些数据，那你可以拉取小型的数据来体验一下全流程。

- 预训练数据的具体详情

1. **Skywork-SkyPile 150B**

Huggingface 开源，包含 150B 中文 tokens，数据集总大小约 620GB，**采用 JSONL 短文本格式储存**。该数据集从公开可访问的中文互联网网页中获取，经过严格的过滤、去重和敏感数据筛除，以确保数据质量。Skywork-SkyPile数据集被开发者描述为目前中文语料社区中最大的数据集、同时也是我们在众多中文预训练任务中都会看到的一个重要数据集，**在本次训练中我们拉取了Skywork-SkyPile的所有数据来使用**。<br><br>
开源网址：[https://huggingface.co/datasets/Skywork/SkyPile-150B/tree/main/data](https://huggingface.co/datasets/Skywork/SkyPile-150B/tree/main/data)

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/14.png)

2. & 8. **wanjuan1.0-nlp**

OpenDataLab开源（**需注册OpenDataLab官网、登录并添加密钥后可访问**）、包含文本数据集、图文数据集、视频数据集三部分，数据总量超过2TB，且已经经过了细粒度的清晰、去重、价值对齐、数据质量较高。其中NLP文本数据集部分包含580G中文数据与440G英文数据，**存储格式为 JSONL 短文本压缩文件格式。在本次训练中我们使用了NLP文本数据集下的所有内容**。

访问网址：[https://opendatalab.org.cn/OpenDataLab/WanJuan1_dot_0](https://opendatalab.org.cn/OpenDataLab/WanJuan1_dot_0) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/16.png)

3. **悟道文本数据集（WuDaoCorporaText）**

BAAI（智源社区）开源（**注册智源社区后即可访问**）、包含100TB原始网页数据中清晰得到的最终数据集、开源200G大小、**存储格式为 Jsonl 短文本**。悟道文本数据集是由北京智源人工智能研究院（BAAI）构建的大规模、高质量中文语料库，旨在支持大型语言模型的训练研究。该数据集从公开可获取的网络资源中收集，经过严格的筛选、去重和清洗，确保数据的多样性和高质量。**在本次训练中，我们使用了WuDao的全部数据，但需要注意的是，这个数据无法通过bash脚本直接从网站拉取、只能通过桌面端下载后再上传到服务器。**

访问网址：[https://data.baai.ac.cn/details/WuDaoCorporaText](https://data.baai.ac.cn/details/WuDaoCorporaText) 


4. **Chinese FineWeb Edu V2**

Huggingface开源、包含约 1.88 亿条记录，总计约 4200 亿个tokens，共计679G左右、以 Parquet 格式存储。Chinese FineWeb Edu V2 是一个面向教育领域自然语言处理（NLP）任务的开源中文预训练数据集，数据集经过显著的优化和扩展和清洗。数据虽然以是否具有足够教育意义评估、但实际上数据来源涵盖多个领域（如 IndustryCorpus2、CCI3、TeleChat 等）。**在本次训练中，我们使用了FineWeb Edu V2的全部数据。**

访问链接：[https://huggingface.co/datasets/opencsg/chinese-fineweb-edu-v2](https://huggingface.co/datasets/opencsg/chinese-fineweb-edu-v2)

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/23.png)

5. **Wikipedia-cn-20230720-filtered**

huggingface开源、包含约 255,000 条记录，总大小约为 1.1 GB，约 85,000 个标记（tokens）、数据结构为JSON，已经过过滤和清洗。Wikipedia-cn-20230720-filtered 是由 Pleisto 发布的开源中文维基百科数据集。

访问网址：[https://huggingface.co/datasets/pleisto/wikipedia-cn-20230720-filtered](https://huggingface.co/datasets/pleisto/wikipedia-cn-20230720-filtered) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/20.png)

6. **BaiduBaike-5.63M**

Huggingface开源、包含约 563 万条百度百科的条目，文件大小约为 16.8 GB、**存储为 JSON 格式**。BaiduBaike-5.63M 是由 Hugging Face 用户 xuqinyang 发布的开源中文百科数据集，由于数据集的条目数量和内容丰富，具体的标记（token）数量可能达到数亿级别。**在本次训练中我们使用了该数据集的全部数据**。

访问网址：[https://huggingface.co/datasets/xuqinyang/BaiduBaike-5.63M](https://huggingface.co/datasets/xuqinyang/BaiduBaike-5.63M) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/21.png)

7. **Zhihu-KOL**

Huggingface开源，该数据集包含约 101 万条知乎问答对，数据总大小约为 1.5 GB，**存储为 Parquet 格式**。Zhihu-KOL 是由 Hugging Face 用户 wangrui6 发布的开源中文问答数据集。

访问网址：[https://huggingface.co/datasets/wangrui6/Zhihu-KOL](https://huggingface.co/datasets/wangrui6/Zhihu-KOL) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/22.png)


9. **SlimPajama-627B**

Huggingface开源（**需提交邮箱地址、以申请访问权限**）、包含6270亿Token，1T左右总数据大小，**存储格式为Jsonl 短文本压缩文件**。SlimPajama-627B 是由 Cerebras 开发的开源数据集，也是TinyLlaMA项目的训练数据集。该数据集通过对 RedPajama 的 1.2 万亿标记（tokens）进行清洗和去重，精简至 6270 亿标记，删除了约 49.6% 的低质量和重复数据。SlimPajama-627B 包含来自多种语料来源的文本，主要以英语为主，经过严格的去重和清洗，以确保数据的高质量和多样性。**在本次训练中，我们使用了SlimPajama全部的数据。**

访问网址：[https://huggingface.co/datasets/cerebras/SlimPajama-627B](https://huggingface.co/datasets/cerebras/SlimPajama-627B) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/18.png)

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/17.png)

10. **Starcoder**

Huggingface开源（**需提交邮箱地址、以申请访问权限**）、包含超过 783GB 的代码，总计约 2,500 亿个tokens，涵盖 86 种编程语言，**采用parquet格式存储**。该数据集是由 BigCode 项目发布的开源代码数据集，旨在用于训练大型代码语言模型，信息来源包括 GitHub 提交记录、问题讨论、和 Jupyter 笔记本等多种数据源。数据经过严格的去重和清洗，以确保高质量和多样性。**在本次训练中，我们拉取了Python、SQL、R、Matlab、JavaScript、Java、Json、C、Rust、Go、TypeScript、Kotlin、Swift、Julia、markdown、html等16种语言进行训练**。<br><br>

开源网址：[https://huggingface.co/datasets/bigcode/starcoderdata](https://huggingface.co/datasets/bigcode/starcoderdata)

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/15.png)

11. **The Stack Dedup**

Huggingface开源（**需提交邮箱地址、以申请访问权限**），3TB大小、覆盖358种编程语言、数据存储方式为Parquet。The Stack Dedup 是由 BigCode 项目发布的开源代码数据集，旨在为代码大型语言模型（Code LLMs）的预训练提供高质量语料、数据来源于具有宽松许可证的开源代码库，经过严格的去重和清洗，以确保数据的高质量和多样性。

访问网址：[https://huggingface.co/datasets/bigcode/the-stack-dedup](https://huggingface.co/datasets/bigcode/the-stack-dedup) 

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/19_.png)

- 未清洗数据

同时，我们曾考虑过使用现在国内已经非常火热的**MNBVC(Massive Never-ending BT Vast Chinese corpus)超大规模中文语料集**，这个数据集是国内爱好者自发收集的、决定制作成最大的中文开源数据集，目前已收集了40T数据，但数据还没有经过校验、质量参差不齐。走这里可以看到当前这个中文数据集的现状：https://github.com/esbatmop/MNBVC

考虑了很久、我们还是没有使用这个不成熟的数据集。在刚收集好、未清洗的巨量文字数据集上进行清洗是一项耗时耗人工的巨大的工作、需要高度自动化的工具和清晰的策略，同时必须结合人工干预以确保高质量结果、这个数据集完成之时、中文预训练模型的质量也将更上一层楼。现在这个数据集的清洗工作将在社区帮助下逐步完成，如果你感兴趣你可以参与。

- 领域数据集

在数据组成中还有许多不成熟的地方，例如我们其实并没有来得及考虑加入一些更专业的领域的数据（金融、医疗、法律、教育、科技等），如果你的目标模型是一个专业领域的模型，那在预训练阶段加入你的专业文本其实对于后续微调也会很有好处！我们尝试在Huggingface以及各类开源社区中收集更专业的数据集、但是得到的大部分领域数据集尺寸都很小，适合微调但并不适合预训练。后续在微调阶段我们会进一步进行介绍。

- 合成数据集

今年以来、合成数据在大模型训练中的应用逐渐普及，尤其是在预训练和微调阶段，在数据集缺乏的领域、或者在对数据集要求特别高的场合、合成数据集展现了其强大的价值、合成数据集训练也成为目前备受关注的领域之一。

**合成数据具有显著的成本优势**，通过使用大模型生成大规模数据，可以显著降低人工标注和数据收集的成本。在稀有场景、多语言任务或特定领域（如医学和法律）中，合成数据能够快速弥补数据空白，增强语料的多样性。此外，合成数据因其高度可控性，可以根据任务需求设计生成策略，用于提高模型在特定场景中的表现。同时，合成数据还能在保护隐私和遵守法规（如 GDPR）方面发挥关键作用，为敏感数据的训练提供替代方案。

然而，依赖合成数据也面临挑战。**合成数据的质量高度依赖生成模型，如果生成模型本身存在偏差或错误，可能导致数据中出现噪声或不准确的内容，从而影响模型的训练效果**。过度依赖合成数据会引发“模式崩塌”问题，即模型的输出缺乏多样性，重复生成相似内容。此外，合成数据可能无法完全反映真实数据分布，容易导致模型在真实应用场景中表现不佳。在伦理层面，合成数据如果模仿真实内容（如代码或文档），可能引发版权和归属争议，尤其是在大规模使用时。

为了更高效地利用合成数据，必须将其与真实数据结合使用，以平衡数据分布，避免模型过拟合到生成模式。对合成数据的质量控制至关重要，可以通过去噪、筛选和验证等手段确保数据的可靠性。此外，在缺乏真实数据的领域，如低资源语言或稀有任务场景，合成数据的作用尤为显著，可以显著提高模型的性能。未来，随着生成技术的发展和质量评估方法的完善，合成数据将更广泛地应用于大模型的训练，但需要开发者在使用中保持审慎，确保其作为真实数据的有益补充而非单一依赖来源。

未来我们会有更多关于合成数据集的内容被讲解、如果你了解更多信息的话，也欢迎让我们知晓！

### 2.2 巨量数据的拉取与存储

在我们使用几个G甚至几十G大小的数据集时，我们可以通过直接下载到本地、或者直接拉取到服务器的方式来调取，但是当数据集变得巨大时、拉取的过程会变得极其漫长且极其不稳定、且每个数据集可能都被原作者或者开源方定义了独特的拉取方式，因此大型数据的拉取是一个相对复杂的问题。在进行预训练之前，我们需要保证我们的所有数据都能够到位。

我们将以巨型中文数据、英文数据、以及代码数据为例、为你讲解多种不同的数据拉取方式、并且在课件中呈现所有数据的拉取流程，你可以选择任意的数据集进行拉取。

| 数据集编号 | 数据集名称                 | 数据属性（中文/英文/代码）                 | 数据量级           | 存储格式            | 是否经过数据清洗 |
|---|----------------------------|-------------------------------------------|--------------------|---------------------|------------------|
| 1| Skywork-SkyPile 150B       | 中文文本                                      | 620GB             | JSONL 短文本        | 是               |
| 2| wanjuan1.0-nlp-CN             | 中文文本                                 | 580GB | JSONL 短文本压缩  | 是               |
| 3| WuDaoCorporaText | 中文文本                                      | 200GB  | JSONL 短文本        | 是               |
| 4| chinese-fineweb-edu-v2 | 75%中文文本，25%英文文本                                      | 670GB  | Parquet        |   是            |
| 5| Wikipedia-CN | 中文文本                                      | 1.1GB  | JSON        |   是            |
| 6| BaiduBaike-5.63M | 中文文本                                      | 17GB  | JSON        |   是            |
| 7| wangrui6/Zhihu-KOL | 中文问答对                                      | 1.5 GB  | parquet        |   是            |
| 8| wanjuan1.0-nlp-EN             | 英文文本                                 | 440GB | JSONL 短文本压缩  | 是               |
| 9| SlimPajama-627B            | 英文为主的混合语言文本                                      | 1TB               | JSONL 短文本压缩    | 是               |
| 10| Starcoder                  | 代码             | 100+GB/768G             | Parquet            | 是               |
| 11| TheStackDedup                 | 代码             | 700+GB/3TB             | Parquet            | 是               |

- **建立拉取数据专用目录**

下面是**命令行代码**，确保你在运行之前已经按照九天老师的视频建立了MateConv虚拟环境，进入了autodl-tmp这个硬盘存储空间，并且建立了MateConv文件夹。当然，你也可以把目录替换成你自己的目录 ↓

```bash
mkdir -p ~/autodl-tmp/MateConv/Data
```

你也可以为某个数据集单独设置目录——

```bash
mkdir -p ~/autodl-tmp/MateConv/Data/SkyPile
```

你可以查看自己的硬盘空间大小、确保自己有给数据准备好足够的空间（在h之后还有一个空格一个点哦）——

```bash
pwd          # 查看当前路径
df -h .      # 查看当前路径所属磁盘的使用情况
```

返回的结果如下 ↓

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/25.png)

这说明还有1.1T的空间供我们使用，你在你自己的磁盘上将会看到相应的空间。你可以在上面的表格中查看具体数据的大小，如果是我们没有标注的数据集，你可以通过Huggingface主页面查看具体的大小，许多数据集都会提到总数据大小，如果没有提到，你也可以通过files页面或者其他页面信息来估算数据总量的大小。

- **设置镜像站**

不同于模型、几乎所有的数据都可以从Huggingface镜像站（[https://hf-mirror.com](https://hf-mirror.com)）进行拉取下载，这样可以避开网络问题、而不用将本地信息代理到线上。因此，在下载数据之前、我们首先要通过设置HF_ENDPOINT环境变量来将我们的下载地址切换到镜像站。

```shell
export HF_ENDPOINT=https://hf-mirror.com     #命令行、设置镜像站环境变量

```

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/24.png)

然后我们有两种方式来拉取数据，一种是使用Huggingface提供的命令行工具`huggingface-cli`来拉取巨型数据、另一种则是可以自定义拉取脚本、结合`aria2c`和`git-lfs`这些工具来更智能、更安全地拉取数据。两种方法分别适用于不同的场景——

| 特性                   | 方法 1：huggingface-cli                           | 方法 2：自定义脚本 + aria2c + git-lfs                       |
|------------------------|--------------------------------------------------|-----------------------------------------------|
| **适用场景**           | 中小型数据集、较小的数据集或一次性下载    | 大型数据集，特别是超大规模数据集、下载耗时长且需要高并发的场景|
| **安装要求**           | 需使用 `huggingface-cli`，但是这一般是huggingface自带的工具           | 需要额外安装 `aria2c` 和 `git-lfs`           |
| **下载速度**           | 受限于单线程速度，可能较慢                        | 支持多线程（如 `-x 16`），下载速度显著提升    |
| **断点续传**           | 理论支持，但大部分实践中无法实现、容易导致重复下载        | 稳定支持，断点续传功能可靠                   |
| **并发支持**           | 不支持                                           | 支持多线程并发下载，提升下载效率             |
| **使用复杂度**         | 简单，官方 CLI 工具，命令较直观                  | 较复杂，**需自定义shell脚本并配置参数**               |
| **镜像支持**           | 支持 `HF_ENDPOINT` 环境变量切换镜像               | 同样支持，可与镜像站搭配使用                 |
| **稳定与适配**             | 对 Hugging Face 官方数据集的兼容性最好<br>但是网络中断或大文件下载时，容易失败                  | 对数据集兼容性较好<br>且在网络不稳定或大文件情况下更稳定             |

#### 2.2.1 huggingface-cli拉取SkyPile数据集

如果你没有安装过huggingface-cli，则需要运行下面的命令行代码👇

```bash
pkgx install huggingface-cli                  #安装huggingface-cli
```

然后你就可以开始拉取数据了！下面帮助你一次性下载完整的数据的命令行代码、请在你的下载目录中准备好至少7-800G的存储空间。<font color="red">**注意！在4MB/s的拉取速度下、下载全部SkyPile数据需要45个小时+7~800G内存，请谨慎运行下面的命令行。**

```bash
huggingface-cli download Skywork/SkyPile-150B --repo-type dataset --resume-download --local-dir ~/autodl-tmp/MateConv/Data/SkyPile  --local-dir-use-symlinks False
```

在这段代码中——
- **`huggingface-cli download`**  调用 Hugging Face 提供的命令行工具，执行下载操作。
  
- **`Skywork/SkyPile-150B`**  指定 Hugging Face 平台上的数据集名称，一般Huggingface上的数据集名称由两部分组成，一个是用户或组织的名称（例如Skywork），另一个是具体的数据集名称（SkyPile-150B）。你需要从Huggingface页面复制正确的名字。

- **`--repo-type dataset`**  明确指定要下载的资源类型是 **数据集**（dataset），如果没有此参数，默认会尝试下载模型（model）。

- **`--resume-download`**  启用断点续传功能。如果之前的下载因网络或其他问题中断，可以从中断处继续下载。但是该功能大部分时候不稳定。

- **`--local-dir ~/autodl-tmp/MateConv/Data/SkyPile`**  指定数据下载后在本地存储的目录路径。在当前代码中，我们是数据将下载到先前建立好的 `~/autodl-tmp/MateConv/Data/SkyPile` 目录。

- **`--local-dir-use-symlinks False`**  关闭符号链接（symlinks）。默认情况下，Hugging Face 会尝试使用符号链接以节省磁盘空间。此参数明确要求不使用符号链接，而是直接将数据下载到指定目录。并且，这一代码现在即将被弃用、因此你可以不写`--local-dir-use-symlinks False`这部分内容。如果你磁盘空间不足、则可以继续尝试开启，并无视掉所有会报的警告。

**如果你想测试一下拉取数据是否成功、则不必一次性拉取全部的数据、而是可以只拉取一部分数据**。我们只需要在原本的代码中间加上特定文件的名字即可👇<font color="red">**对于SkyPile数据拉取一条数据大约需要10min，网速快的情况下可以在5mins内拉取完**——

```bash
huggingface-cli download Skywork/SkyPile-150B data/2020-40_zh_head_0000.jsonl --repo-type dataset --resume-download --local-dir ~/autodl-tmp/MateConv/Data/SkyPile  --local-dir-use-symlinks False
```

在这段命令中，我们只需要在原本的数据集名称后面**增加一个空格、并街上我们要下载的单一文件的目录**，就可以下载单一文件了。同样的，如果我们要拉取多个文件，则可以按照**空格 + 要下载的文件目录**的方式排列相应的代码。

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/27.png)

数据的目录具体是什么？你可以在Huggingface页面的files页面下找到👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/28.png)

找到文件目录后、点击相应的具体要下载的文件👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/29.png)

进行拉取时，你将看到这样的进度条👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/30.png)

拉取完成之后，你将可以在你设置的线上目录下找到相应的数据👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/31.png)

#### 2.2.2 多线程并行拉取SkyPile数据集

使用`huggingface-cli`拉取数据固然简单，但是最大的困境就在于拉取时间太长、极其容易断线、而且网速极其不稳定。因此针对大型数据、我们要使用Huggingface镜像站官方提供的hfd脚本、并结合`aria2c`和`git-lfs`来进行**多线程并行拉取**。

`hfd.sh` 是由 Hugging Face 镜像站开发的一款专用下载工具，专为 Hugging Face 平台的模型和数据集提供稳定、高速的下载支持。它基于成熟的多线程下载工具 Aria2 构建，结合 Hugging Face 的资源特点，实现了多源分块下载和断点续传功能，能够有效解决大文件下载中的中断问题。通过 `hfd.sh`，用户可以灵活选择下载模型或数据集，并支持包括文件筛选（`--include` 和 `--exclude`）、多线程设置（`-x`）和并行任务配置（`-j`）在内的高度自定义参数，适用于高效管理 Hugging Face 平台上的资源下载需求。

在使用hfd时、还需要单独安装支持库aria2c和git-lfs。其中 Aria2 是一个轻量级、多协议的、**支持多线程分块下载、文件分块下载、多元下载、多服务器下载的数据并行下载工具**，Aria2 可以通过命令行运行，适合集成到脚本中进行自动化下载，且其丰富的参数配置允许用户高度自定义，包括限速、连接数、分块大小等。此外，Aria2 的资源占用极低，即使在性能较低的设备上也能顺畅运行，因而成为开发者和自动化任务中常用的下载工具之一。

而 Git LFS（Git Large File Storage）是 Git 的一种扩展工具，用于优化对大文件的版本控制和存储管理。**它通过将大文件（如音视频文件、图像、模型权重等）存储在外部专用的文件存储中，而不是直接保存在 Git 仓库中，从而避免仓库膨胀问题**。Git LFS 用指针文件替代大文件在版本控制系统中的实际存储，当需要使用这些大文件时，Git LFS 会在后台自动下载和管理它们，确保开发者的操作与普通 Git 工作流无缝兼容。其优点在于有效减少 Git 仓库的体积，提升拉取、克隆和推送等操作的效率，是处理数据密集型项目（如机器学习、音视频编辑等）时的重要工具。

这三个工具结合、可以大幅度提升数据和模型的下载速度和稳定性、可以为开发者提供便捷且高效的下载体验。

接下来我们来看一下首次使用hfd脚本的流程——

**下载hfd.sh文件**、并赋予直接修改命令的权限——

```bash
wget https://hf-mirror.com/hfd/hfd.sh    #下载hfd.sh文件到当前的目录
chmod a+x hfd.sh                         #文件hfd.sh被赋予执行权限
```

**安装aria2c**——

```bash
sudo apt update                            #更新软件包索引
sudo apt install aria2                     #安装aria2c
```

**安装完毕后、需要打印版本号以验证成功**——

```bash
aria2c --version                           #打印版本号
```

出版本号则说明安装成功👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/34.png)

**安装git-lfs**——

```bash
sudo apt install git-lfs                   #安装git-lfs
```

**安装完毕后、打印版本号以验证成功**——

```bash
git lfs --version                         #打印版本号
```

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/33.png)

现在可以开始使用`hfd脚本` + `aria2c` + `git-lfs`来拉取数据了。<font color="red">**注意！在4MB/s的拉取速度下、下载下面全部SkyPile数据需要6\~7个小时+7~800G内存，请谨慎运行下面的命令行。**

```bash
./hfd.sh Skywork/SkyPile-150B --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/SkyPile
```

在这段代码中——

- **`./hfd.sh`**：调用本地的 Bash 脚本 `hfd.sh`，用来从 Hugging Face 平台下载指定的资源。`hfd.sh` 是一个辅助工具，封装了 Hugging Face 的下载逻辑，并支持更多高级功能（如断点续传、多线程、多任务下载）。

- **`Skywork/SkyPile-150B`**：指定 Hugging Face 数据集的 **完整名称**，由组织或用户名称（Skywork）和具体数据集名称（SkyPile-150B）组成。

- **`--dataset`**：指明要下载的是一个 **数据集**（dataset），而不是模型（model）。

- **`--tool aria2c`**：指定使用的下载工具，这里是 `aria2c`，相比于 `wget` 或其他工具，`aria2c` 支持分块下载和多线程，能显著提升下载速度。

- **`-x 10`**：表示设置 **线程数** 为 10，线程数决定了单个文件的下载并发数，`aria2c` 会分块并同时下载多个部分。`-x` 的最大值通常受hfd脚本或工具限制，如果超过限制会报错👇例如设置了-x为16、

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/35.png)

**`--local-dir ~/autodl-tmp/MateConv/Data/SkyPile`**：指定下载文件存储的本地路径。在这里，文件将存储在 `~/autodl-tmp/MateConv/Data/SkyPile` 目录中。如果目录不存在，脚本会尝试自动创建。

同样的、**如果你想测试一下拉取数据是否成功、则不必一次性拉取全部的数据、而是可以只拉取一部分数据**。我们只需要在原本的代码中间加上`--include` + 特定文件的名字即可👇<font color="red">**在4MB/s网络下、对于SkyPile数据拉取一条数据大约需要1mins，网速快的情况下可以在30s内拉取完**——

```bash
./hfd.sh Skywork/SkyPile-150B --dataset --include data/2020-40_zh_head_0001.jsonl --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/SkyPile
```

- **`--include data/2020-40_zh_head_0001.jsonl`**：表示只下载数据集中的指定文件路径。在 `Skywork/SkyPile-150B` 数据集中，这个路径为 `data/2020-40_zh_head_0000.jsonl`。该参数允许使用通配符（如 `*.jsonl`）来匹配多个文件。

> **`hfd脚本`**

作为Hugging Face 镜像站开发的专用下载工具、hfd脚本提供了巨大的可以修改的空间、几乎每个板块都有可以修改的流程，我们重点要关注的有——

- <font color="red">**参数初始化Part**
```bash
REPO_ID=$1
shift

# Default values
TOOL="aria2c"
THREADS=4
CONCURRENT=5
HF_ENDPOINT=${HF_ENDPOINT:-"https://huggingface.co"}
INCLUDE_PATTERNS=()
EXCLUDE_PATTERNS=()
REVISION="main"
```

- **解释**：**这段代码位于整个hfd脚本的开头**。
  - `REPO_ID=$1`：读取第一个参数，作为 Hugging Face 的数据集或模型 ID。
  - `shift`：移除第一个参数，便于解析后续参数。
  - **默认参数**：
    - `TOOL="aria2c"`：默认使用 `aria2c` 作为下载工具。
    - `THREADS=4`：每个文件下载的默认线程数为 4。
    - `CONCURRENT=5`：同时并行下载的文件数为 5。
    - `HF_ENDPOINT`：默认使用 Hugging Face 的主域名 `https://huggingface.co`，<font color="green">**但我们之前已经通过修改环境变量将HF_ENDPOINT这个变量改变为Huggingface镜像站了。**</font>
    - `REVISION="main"`：默认下载主分支内容。

---

- <font color="red">**参数验证和解析Part**
```bash
validate_number() {
    [[ "$2" =~ ^[1-9][0-9]*$ && "$2" -le "$3" ]] || { printf "${RED}[Error] $1 must be 1-$3${NC}\n"; exit 1; }
}

while [[ $# -gt 0 ]]; do
    case $1 in
        --include) ... ;;
        --exclude) ... ;;
        --tool) ... ;;
        -x) validate_number "threads (-x)" "$2" 10; THREADS="$2"; shift 2 ;; 
        -j) validate_number "concurrent downloads (-j)" "$2" 10; CONCURRENT="$2"; shift 2 ;; 
        --dataset) DATASET=1; shift ;;
        --local-dir) LOCAL_DIR="$2"; shift 2 ;;
        --revision) REVISION="$2"; shift 2 ;;
        *) display_help ;;
    esac
done
```

- **解释**：这段代码位于整个脚本的前半段、位于参数初始化part的后面。<br><br>
  - `validate_number`：验证参数是否为正整数，且不超过指定最大值。例如：<br><br>
    - `-x`：线程数（最大 10）。将数据分块、分到不同线程上进行下载。<font color="green">**如果你想要修改最大线程数、可以将`-x) validate_number "threads (-x)" "$2" 10; THREADS="$2"; shift 2 ;;`这行中的10进行修改**。</font>
    > - 一般来说、最大线程数会受到Huggingface等公共资源、以及CPU性能的限制，通常来说我们会将线程数设置在4-8之间（设备比较陈旧、或者设备有限就设置为4）、但最大可以设置到16。<br>
    - `-j`：并行下载数（最大 10）。同步下载多个文件。<font color="green">**如果想要修改并行下载数，可以将`-j) validate_number "concurrent downloads (-j)" "$2" 10; CONCURRENT="$2"; shift 2 ;;`这行中的10进行修改**。</font>
    > - 如果单个文件较大、增加线程数会比较明智、如果是下载多个小文件、增加并行下载数才可以更快完成。<br>
  - `case $1`：解析传入参数，支持以下选项：
    - `--include`：指定下载文件的匹配规则。
    - `--exclude`：排除匹配的文件。
    - `--tool`：选择下载工具（`aria2c` 或 `wget`）。
    - `--local-dir`：指定文件存储路径。
    - `--revision`：选择下载的分支版本。
<br><br>

**该如何选择具体的线程？**

| 场景                          | 推荐 `-x`（线程数） | 推荐 `-j`（并行数） |
|-------------------------------|---------------------|---------------------|
| 普通个人电脑（带宽 ≤ 50Mbps）  | 4-6                 | 2-4                 |
| 高性能电脑（带宽 ≥ 100Mbps）   | 8-16                | 5-8                 |
| 服务器或高速网络（带宽 ≥ 1Gbps）| 16-32               | 10-16               |
| 小文件为主                   | 2-4                 | 8-10                |
| 大文件为主                   | 8-16                | 2-5                 |

需要注意的是、`-x`（线程数）和 `-j`（并行下载数）会同时消耗资源，不建议都设置为过高值。如果 `-x=16`，建议 `-j` 限制在 4-5 左右，反之亦然。大多数情况下、`-x 8`，`-j 4` 是合理起点。高性能设备则可以适当提高到 `-x 16`，`-j 10`。
<br><br>
**你可以使用aria2c测试线程的情况**、测试后根据设备和网络的情况进一步优化，找到适合你的配置。

> 单文件测试线程数（`-x`）、下载大文件时测试：
  ```bash
  aria2c -x 4 https://example.com/large-file.zip
  ```
  - 尝试增加 `-x` 的值，如 8、16，观察是否有明显提速。

> 多文件测试并行数（`-j`）、下载多个小文件时测试：
  ```bash
  aria2c -x 4 -j 8 -i file-list.txt
  ```
  - 增加 `-j` 的值，如 10、16，观察下载时间是否减少。
<br><br>

**除此之外、你还可以通过代码来查看你CPU支持的最大线程数——**

```bash
grep -c processor /proc/cpuinfo          #显示现在CPU上的总线程数
```

输出——

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/36.png)

虽然CPU支持的最大线程是128、但是实际上Huggingface mirror镜像站最大只能支持16。

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/37.png)

---

- **文件过滤Part**
```bash
INCLUDE_REGEX=$(printf '%s\n' "${INCLUDE_PATTERNS[@]}" | sed 's/\./\\./g; s/\*/.*/g' | paste -sd '|' -)
EXCLUDE_REGEX=$(printf '%s\n' "${EXCLUDE_PATTERNS[@]}" | sed 's/\./\\./g; s/\*/.*/g' | paste -sd '|' -)
```

- **解释**：
  - 将 `--include` 和 `--exclude` 的文件模式转换为正则表达式，用于筛选需要下载的文件。这样可以实现灵活的文件选择，避免下载整个数据集或模型，提高效率。

---

- **aria2c文件下载Part**
```bash
aria2c --console-log-level=error --file-allocation=none -x "$THREADS" -j "$CONCURRENT" -s "$THREADS" -k 1M -c -i "$fileslist_file"
```

- **解释**：
  - 使用 `aria2c` 下载文件：
    - `-x "$THREADS"`：每个文件的下载线程数、和之前代码中设置的一样。
    - `-j "$CONCURRENT"`：并行下载的文件数、和之前代码中设置的一样。
    - `-s "$THREADS"`：分块下载数、和之前代码中设置的一样。
    - `-k 1M`：设置分块大小为 1MB，如果你的带宽较大、使用更大的分块（比如4M或者8M）能够更充分地发挥网络性能，如果你的网络不稳定你可以设置到1MB或者512kb。
    - `-i "$fileslist_file"`：指定文件列表作为输入。

---

这段脚本支持文件筛选、授权访问、多线程并行下载等功能，适合高效下载 Hugging Face 数据集或模型。你可以通过调整 `-x`、`-j` 等参数优化下载速度，也可以通过 `--include` 精确控制下载的文件。

#### 2.2.3 其他Huggingface数据集的拉取

| 数据集编号 | 数据集名称                 | 数据属性（中文/英文/代码）                 | 数据量级           | 存储格式            | 是否经过数据清洗 |
|---|----------------------------|-------------------------------------------|--------------------|---------------------|------------------|
| 1| Skywork-SkyPile 150B       | 中文文本                                      | 620GB             | JSONL 短文本        | 是               |
| 2| wanjuan1.0-nlp-CN             | 中文文本                                 | 580GB | JSONL 短文本压缩  | 是               |
| 3| WuDaoCorporaText | 中文文本                                      | 200GB  | JSONL 短文本        | 是               |
| 4| chinese-fineweb-edu-v2 | 75%中文文本，25%英文文本                                      | 670GB  | Parquet        |   是            |
| 5| Wikipedia-CN | 中文文本                                      | 1.1GB  | JSON        |   是            |
| 6| BaiduBaike-5.63M | 中文文本                                      | 17GB  | JSON        |   是            |
| 7| wangrui6/Zhihu-KOL | 中文问答对                                      | 1.5 GB  | parquet        |   是            |
| 8| wanjuan1.0-nlp-EN             | 英文文本                                 | 440GB | JSONL 短文本压缩  | 是               |
| 9| SlimPajama-627B            | 英文为主的混合语言文本                                      | 1TB               | JSONL 短文本压缩    | 是               |
| 10| Starcoder                  | 代码             | 100+GB/768G             | Parquet            | 是               |
| 11| TheStackDedup                 | 代码             | 700+GB/3TB             | Parquet            | 是               |

剩下Huggingface数据集、包括opencsg/chinese-fineweb-edu-v2、cerebras/SlimPajama-627B、wangrui6/Zhihu-KOL、xuqinyang/BaiduBaike-5.63M以及pleisto/wikipedia-cn-20230720-filtered数据集，都可以通过和SkyPile类似的方式拉取。具体流程如下👇

1. **设置Huggingface镜像站变量、并确保下载好hfd、设置好目录文件**

```shell
export HF_ENDPOINT=https://hf-mirror.com            #命令行、设置镜像站环境变量

wget https://hf-mirror.com/hfd/hfd.sh               #下载hfd.sh文件到当前的目录
chmod a+x hfd.sh                                    #文件hfd.sh被赋予执行权限

sudo apt update                                     #更新软件包索引
sudo apt install aria2                              #安装aria2c

sudo apt install git-lfs                            #安装git-lfs

#设置目录文件
mkdir -p ~/autodl-tmp/MateConv/Data/Fineweb         
mkdir -p ~/autodl-tmp/MateConv/Data/wikicn       
mkdir -p ~/autodl-tmp/MateConv/Data/baidubaike       
mkdir -p ~/autodl-tmp/MateConv/Data/zhihu   
mkdir -p ~/autodl-tmp/MateConv/Data/slimpajama       
```

2. **`hfd脚本` + `aria2c` + `git-lfs`多线程并行拉取所有数据**

- Fineweb

一次性拉取所有数据的命令👇<font color="red">**注意！在4MB/s、10个线程并行拉取情况下、下载全部fineweb数据需要6\~7小时时间+7\~800G内存，请谨慎运行下面的命令行。**

```bash
./hfd.sh opencsg/chinese-fineweb-edu-v2 --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Fineweb
```

单独拉取一个文件的命令👇在10个线程并行拉取情况下、要大约1~2min左右的时间——

```bash
./hfd.sh opencsg/chinese-fineweb-edu-v2 --dataset --include data/00000.parquet --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Fineweb
```

- SlimPajama

一次性拉取所有数据的命令👇<font color="red">**注意！在4MB/s、10个线程并行拉取情况下、下载全部SlimPajama数据需要个8\~9小时+900G以上内存，请谨慎运行下面的命令行。**

```bash
./hfd.sh cerebras/SlimPajama-627B --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/slimpajama
```

单独拉取一个文件的命令👇zst文件非常小，在10个线程并行拉取情况下、要大约3s左右的时间——

```bash
./hfd.sh cerebras/SlimPajama-627B --dataset --include train/chunk1/example_train_0.jsonl.zst --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/slimpajama
```

- 小型数据群

其中，zhihu数据大约1.5G、wiki数据大约1G、下载时间都很短、百度百科数据大约17G，下载时间会略长一些，但在多线程并行下速度都不是问题。

```bash
#zhihu
./hfd.sh wangrui6/Zhihu-KOL --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/zhihu

#Baidubaike
./hfd.sh xuqinyang/BaiduBaike-5.63M --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/baidubaike

#Wiki
./hfd.sh pleisto/wikipedia-cn-20230720-filtered --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/wikicn

```

下载好之后，可以在相应的目录下找到你的数据👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/50.png)

#### 2.2.4 万卷中英文数据集的拉取

万卷数据集来自Openxlab实验室，因此需要走Openxlab实验室的流程进行下载。

**1. 在Opendatalab首页进行注册、获取自己的Access Key/Secret Access Key**

在这里进行手机号注册 → https://opendatalab.org.cn/OpenDataLab/WanJuan1_dot_0

注册完毕后到这里领取自己的AK/SK → https://sso.openxlab.org.cn/usercenter

> step1——

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/38.png)

> step 2——

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/39.png)

2. **安装openxlab、进行openxlab登录**

> step 1——

```bash
  pip install openxlab #安装
  pip install -U openxlab #版本升级
```

> step 2——

```bash
  openxlab login
```

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/40.png)

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/42.png)

3. **设置数据下载目录、查看数据集线上详情**

```bash
mkdir -p ~/autodl-tmp/MateConv/Data/wanjuan

openxlab dataset info --dataset-repo OpenDataLab/WanJuan1_dot_0 #数据集信息及文件列表查看
```

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/43.png)

4. **数据下载**

<font color="red">**注意！wanjuan数据下载是自动多线程并行的、然而在平均网速32MB/s情况下、下载中文+英文nlp方向的wanjuan数据需要12小时+1.5T以上内存，请谨慎运行下面的命令行。**

```bash
#get命令下载整个数据集
openxlab dataset get --dataset-repo OpenDataLab/WanJuan1_dot_0 --source-path /raw/nlp --target-path ~/autodl-tmp/MateConv/Data/wanjuan
```

<font color="red">**注意！在平均网速32MB/s情况下、下载1个4G左右的压缩文件大约需要2分钟时间，你依然可以用下面的代码来测试、流程通了之后再执行完整命令。**

```bash
#download命令特定数据集文件下载
openxlab dataset download --dataset-repo OpenDataLab/WanJuan1_dot_0 --source-path /raw/nlp/CN/ChinaNews-cn/part-006853-a894b46e.jsonl.tar.gz --target-path ~/autodl-tmp/MateConv/Data/wanjuan 
```

- **`openxlab dataset download`**：调用 OpenXLab 提供的命令行工具，用于管理和操作 OpenXLab 平台上的数据。**`dataset download`**：表示操作类型为下载数据集文件。

- **`--dataset-repo OpenDataLab/WanJuan1_dot_0`**：**`--dataset-repo`**：指定目标数据集的仓库名称，和huggingface一样，会包括数据集所属的组织或用户名、同时还会包括数据集本身的名称。你可以在这里找到数据集的仓库全名 ↓

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/45.png)

- **`--source-path /raw/nlp/CN/ChinaNews-cn/part-006853-a894b46e.jsonl.tar.gz`**：**`--source-path`**：指定要下载的数据集中的具体文件路径，你可以在这里找到相应的路径。你可以指向一个文件夹、也可以指向具体的数据集文件 ↓
       
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/46.png)

- **`--target-path ~/autodl-tmp/MateConv/Data/wanjuan`**：指定下载后文件的存储位置。

开始下载之后，会有自动打印和监控流程——

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/44.png)

下载完毕后，你将可以在finalshell的目录中查看到文件👇

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/47.png)

下载完成后、需要进行解压、先单独设置解压后的文件目录——

```bash
mkdir -p ~/autodl-tmp/MateConv/Data/wanjuan/clean            #单独设置解压后的文件目录
```

wanjuan官方提供了解压用的py脚本，可以直接上传到服务器进行使用👇也可直接在线上jupyter进行使用。

```python
import glob 
import os
import time
if __name__ == "__main__":
    dir_list = glob.glob("/data1/step0_rawdata/wanjuan/**/*.jsonl.tar.gz", recursive=True)
    target_root_dir = "~/autodl-tmp/MateConv/Data/wanjuan/clean"
    print(dir_list)
    print(len(dir_list))
    for coutner, file_dir in enumerate(dir_list):
        print(file_dir)
        t0 = time.time()
        save_need_name = file_dir.split("/")[-3:]
        tmp_name = save_need_name[-1]
        # 解压后的文件名称
        tmp_name = tmp_name.replace(".jsonl.tar.gz", ".jsonl")
        tmp_name = os.path.join(target_root_dir, tmp_name)
        # 目标名称带着文件夹名
        save_need_name[-1] = save_need_name[-1].split(".")[0]
        target_file_name = "_".join(save_need_name)
        target_file_name += ".jsonl"
        target_file_name = os.path.join(target_root_dir, target_file_name)
        print("old_name:", tmp_name)
        print("new name:",target_file_name)
        if os.path.exists(target_file_name):
            print(f"have: {target_file_name}")
        cmd = f"tar -zxvf {file_dir} -C {target_root_dir}"
        os.system(cmd)
        os.system(f"mv {tmp_name} {target_file_name}")
        print(coutner, time.time()-t0)
    
    en_dir = os.path.join(target_root_dir, "wanjuan_en")
    cn_dir = os.path.join(target_root_dir, "wanjuan_zh")
    os.mkdir(en_dir)
    os.mkdir(cn_dir)
    mv_cmd1 = f"mv {target_root_dir}/EN*.jsonl {en_dir}"
    os.system(mv_cmd1)
    mv_cmd2 = f"mv {target_root_dir}/CN*.jsonl {cn_dir}"
    os.system(mv_cmd2)

```

同时需要注意的是，<font color="green">**由于wanjuan数据集在下载的时候需要登录到Openxlab，因此在开始下载其他数据集之前、可以尝试重开Bash、并需要重新设置其他数据集所需要的下载环境变量、镜像站等信息**。

#### 2.2.5 WuDao数据集的拉取

**WuDao数据集不支持命令行方式下载、因此只能从网页下载并上传到线上服务器**。幸运的是WuDao数据集本身不是很大、只需要关注中间断线情况、耐心下载即可。

1. **进入页面、微信登录注册**：https://data.baai.ac.cn/details/WuDaoCorporaText

2. **网页进行下载、也可以使用迅雷等工具👇**

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/48.png)

3. **上传至服务器👇由于autodl上传有限速，因此推荐和阿里云盘结合使用、可大幅提升上传速度**

具体autodl链接阿里云盘文件参考👉https://www.autodl.com/docs/netdisk/

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/49.png)

#### 2.2.6 代码类数据集的拉取

| 数据集编号 | 数据集名称                 | 数据属性（中文/英文/代码）                 | 数据量级           | 存储格式            | 是否经过数据清洗 |
|---|----------------------------|-------------------------------------------|--------------------|---------------------|------------------|
| 1| Skywork-SkyPile 150B       | 中文文本                                      | 620GB             | JSONL 短文本        | 是               |
| 2| wanjuan1.0-nlp-CN             | 中文文本                                 | 580GB | JSONL 短文本压缩  | 是               |
| 3| WuDaoCorporaText | 中文文本                                      | 200GB  | JSONL 短文本        | 是               |
| 4| chinese-fineweb-edu-v2 | 75%中文文本，25%英文文本                                      | 670GB  | Parquet        |   是            |
| 5| Wikipedia-CN | 中文文本                                      | 1.1GB  | JSON        |   是            |
| 6| BaiduBaike-5.63M | 中文文本                                      | 17GB  | JSON        |   是            |
| 7| wangrui6/Zhihu-KOL | 中文问答对                                      | 1.5 GB  | parquet        |   是            |
| 8| wanjuan1.0-nlp-EN             | 英文文本                                 | 440GB | JSONL 短文本压缩  | 是               |
| 9| SlimPajama-627B            | 英文为主的混合语言文本                                      | 1TB               | JSONL 短文本压缩    | 是               |
| 10| Starcoder                  | 代码             | 100+GB/768G             | Parquet            | 是               |
| 11| TheStackDedup                 | 代码             | 700+GB/3TB             | Parquet            | 是               |

- **权限申请**

代码数据一般都是量非常大的数据、其特点不仅在于数据量大、还需要我们规定特定的编程语言进行下载，因此CLI代码会与传统文字数据集略有差别。首先，代码数据集的下载会需要权限、因此我们要先申请相应的access token——

1. **挂上梯子、注册Huggingface账号**👉https://huggingface.co/

2. **登录后、在个人profile页面找到自己的username**

> - 点击右上角——
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/51.png)

> - 复制username——
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/52.png)

3. **登录后、在这个页面下建立属于自己的AccessToken**👉https://huggingface.co/settings/tokens

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/53.png)

4. **在下面的页面中、建立token名字、并且为所需要的数据集申请权限**

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/54_.png)

点击create token、获得token后直接复制，**你将不会有第二次复制token的机会、因此务必要在这个时候复制token**。

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/56.png)

token建好后，可以在token页面中看到、但此时你只能看到你的token名称、将不能再复制具体的access token码了。

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/57.png)

---

- **jq工具配置**

与文字数据不同的是、代码数据本身的结构比较复杂、元数据的JSON文件可能嵌套了更多层次结构（例如文件分组、标签、语言分类等），因此解析起来会更为缓慢。为此，我们要下载`jq`工具来帮助我们在下载过程中更高效地解析代码数据。

`jq` 是一个专为处理 JSON 数据设计的轻量级命令行工具，能够高效解析、查询、修改和格式化 JSON 文件或数据流。它的语法灵活，支持强大的过滤和数据操作功能，类似于 JSON 格式的 `sed` 或 `awk`。`jq` 的优势在于其速度快、占用资源少，并能以简洁的方式处理复杂的 JSON 结构，而无需编写冗长的代码。安装 `jq` 的主要原因是许多现代工具（如 `hfd.sh`）在处理大型 JSON 文件时可以借助它快速解析元数据，显著提高文件筛选和匹配的效率，从而避免使用较慢的替代方案（如 `grep` 或 `awk`）。对于需要处理 JSON 数据的开发者和运维人员来说，`jq` 是不可或缺的工具之一。

下面是配置`jq`库的命令行代码👇

```bash
sudo apt update                                  #更新软件包索引
sudo apt install jq -y                           #安装jq

jq --version                                     #检查版本号
```

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/58.png)

---

- **数据拉取**

配置好jq工具后，可以开始进行正式的数据下载了！首先还是要设置好相应的目录——

1. **设置Huggingface镜像站变量、并确保下载好hfd、设置好目录文件**

```shell
export HF_ENDPOINT=https://hf-mirror.com            #命令行、设置镜像站环境变量

wget https://hf-mirror.com/hfd/hfd.sh               #下载hfd.sh文件到当前的目录
chmod a+x hfd.sh                                    #文件hfd.sh被赋予执行权限

sudo apt update                                     #更新软件包索引
sudo apt install aria2                              #安装aria2c

sudo apt install git-lfs                            #安装git-lfs

#设置目录文件
mkdir -p ~/autodl-tmp/MateConv/Data/Starcoder   
mkdir -p ~/autodl-tmp/MateConv/Data/TheStackDedup           
```

2. **`hfd脚本` + `aria2c` + `git-lfs`多线程并行拉取数据**

- **StarCoder**

一次性拉取所有数据的命令👇<font color="green">**注意！你需要将下面的hf_token修改为你刚才申请的有权限的token、同时将hf_username修改为你自己的huggingface账户的名字**。

<font color="red">**同时，在均32MB/s速度下、下载全部starcoder数据需要6\~7小时时间+800G以上内存，请谨慎运行下面的命令行。**

```bash
./hfd.sh bigcode/starcoderdata --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Starcoder --hf_token xxx --hf_username TsaiTsai0929
```

单独拉取一种语言的命令👇<font color="red">**例如Python、大约22G的数据，在34MB/s速度下、需要大约11~12min左右的时间**——

```bash
./hfd.sh bigcode/starcoderdata --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Starcoder --include *python --hf_token xxx --hf_username TsaiTsai0929
```

当前的hfd脚本一次只能支持一个目录、但是对代码数据集我们必然会需要不止一种代码。**在本次训练中，我们拉取了Python、SQL、R、Matlab、JavaScript、Java、Json、C、Rust、Go、TypeScript、Kotlin、Swift、Julia、markdown、html等16种语言进行训练**。因此你可以设置这样的CLI代码、一次性运行👇

```bash
./hfd.sh bigcode/starcoderdata --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Starcoder --include *sql* --hf_token xxx --hf_username TsaiTsai0929

./hfd.sh bigcode/starcoderdata --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Starcoder --include *r* --hf_token xxx --hf_username TsaiTsai0929

./hfd.sh bigcode/starcoderdata --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/Starcoder --include *matlab* --hf_token xxx --hf_username TsaiTsai0929

```

- **TheStackDedup**

一次性拉取所有数据的命令👇<font color="green">**注意！你需要将下面的hf_token修改为你刚才申请的有权限的token、同时将hf_username修改为你自己的huggingface账户的名字**。

<font color="red">**同时，在均32MB/s速度下、下载全部starcoder数据需要24小时时间+3T以上内存，请谨慎运行下面的命令行。**

```bash
./hfd.sh bigcode/the-stack-dedup --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/TheStackDedup --hf_token xxx --hf_username TsaiTsai0929
```

单独拉取一种语言的命令👇<font color="red">**例如sql、大约4G的数据，在34MB/s速度下、需要大约3~4min左右的时间**——

```bash
./hfd.sh bigcode/the-stack-dedup --dataset --tool aria2c -x 10 --local-dir ~/autodl-tmp/MateConv/Data/TheStackDedup --include *sql --hf_token xxx --hf_username TsaiTsai0929
```

至此、所有数据都拉取完毕了！下载好所有的数据后、你可能发现了、每个数据的格式、状态、结构都不尽相同，因此**我们还需要针对每一个数据进行特定的预处理**。与下载一样、对这些数据进行预处理的流程大多需要分布式工具来辅助我们，接下来让我们一起看看巨量数据的预处理流程。

| 数据集编号 | 数据集名称                 | 数据属性（中文/英文/代码）                 | 数据量级           | 存储格式            | 是否经过数据清洗 |
|---|----------------------------|-------------------------------------------|--------------------|---------------------|------------------|
| 1| Skywork-SkyPile 150B       | 中文文本                                      | 620GB             | JSONL 短文本        | 是               |
| 2| wanjuan1.0-nlp-CN             | 中文文本                                 | 580GB | JSONL 短文本压缩  | 是               |
| 3| WuDaoCorporaText | 中文文本                                      | 200GB  | JSONL 短文本        | 是               |
| 4| chinese-fineweb-edu-v2 | 75%中文文本，25%英文文本                                      | 670GB  | Parquet        |   是            |
| 5| Wikipedia-CN | 中文文本                                      | 1.1GB  | JSON        |   是            |
| 6| BaiduBaike-5.63M | 中文文本                                      | 17GB  | JSON        |   是            |
| 7| wangrui6/Zhihu-KOL | 中文问答对                                      | 1.5 GB  | parquet        |   是            |
| 8| wanjuan1.0-nlp-EN             | 英文文本                                 | 440GB | JSONL 短文本压缩  | 是               |
| 9| SlimPajama-627B            | 英文为主的混合语言文本                                      | 1TB               | JSONL 短文本压缩    | 是               |
| 10| Starcoder                  | 代码             | 100+GB/768G             | Parquet            | 是               |
| 11| TheStackDedup                 | 代码             | 700+GB/3TB             | Parquet            | 是               |

### 2.3 巨量数据清洗与数据预处理